# Description

In this notebook we prepare the simulation data of Li-ion battery under the constant current load. Unstable simulations are not saved.

# Import

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from prog_models.models import BatteryElectroChem
import sys
sys.path.append('../')

from copy import deepcopy
from functools import partial

import warnings
warnings.filterwarnings(action='error', category=RuntimeWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)

number_of_trajectories = 10

# Simulate

In [3]:
def update(ps):
    """
    Updates the battery model with the new parameters (This will be deprecated when the github repo is updated)
    """
    ps['qMax'] = ps['qMobile']/(ps['xnMax']-ps['xnMin']) # note qMax = qn+qp
    # Volumes (total volume is 2*P.Vol), assume volume at each electrode is the
    # same and the surface/bulk split is the same for both electrodes
    ps['VolS'] = ps['VolSFraction']*ps['Vol'] # surface volume
    ps['VolB'] = ps['Vol'] - ps['VolS'] # bulk volume

    # set up charges (Li ions)
    ps['qpMin'] = ps['qMax']*ps['xpMin'] # min charge at pos electrode
    ps['qpMax'] = ps['qMax']*ps['xpMax'] # max charge at pos electrode
    ps['qpSMin'] = ps['qMax']*ps['xpMin']*ps['VolSFraction'] # min charge at surface, pos electrode
    ps['qpBMin'] = ps['qMax']*ps['xpMin']*(ps['Vol'] - ps['VolS'])/ps['Vol'] # min charge at bulk, pos electrode
    ps['qpSMax'] = ps['qMax']*ps['xpMax']*ps['VolS']/ps['Vol'] # max charge at surface, pos electrode
    ps['qpBMax'] = ps['qMax']*ps['xpMax']*ps['VolB']/ps['Vol'] # max charge at bulk, pos electrode
    ps['qnMin'] = ps['qMax']*ps['xnMin'] # max charge at neg electrode
    ps['qnMax'] = ps['qMax']*ps['xnMax'] # max charge at neg electrode
    ps['qnSMax'] = ps['qMax']*ps['xnMax']*ps['VolSFraction'] # max charge at surface, neg electrode
    ps['qnBMax'] = ps['qMax']*ps['xnMax']*(1-ps['VolSFraction']) # max charge at bulk, neg electrode
    ps['qnSMin'] = ps['qMax']*ps['xnMin']*ps['VolSFraction'] # min charge at surface, neg electrode
    ps['qnBMin'] = ps['qMax']*ps['xnMin']*(1-ps['VolSFraction']) # min charge at bulk, neg electrode
    ps['qSMax'] = ps['qMax']*ps['VolSFraction'] # max charge at surface (pos and neg)
    ps['qBMax'] = ps['qMax']*(1-ps['VolSFraction']) # max charge at bulk (pos and neg)
    ps['x0']['qpS'] = ps['qpSMin']
    ps['x0']['qpB'] = ps['qpBMin']
    ps['x0']['qnS'] = ps['qnSMax']
    ps['x0']['qnB'] = ps['qnBMax']
    ps['x0']['qMax'] = ps['qMobile']
    ps['x0']['Ro'] = ps['Ro']
    return ps

In [4]:
q = np.random.uniform(7500, 7600)
R0 = np.random.uniform(0.117215-0.01, 0.117215+0.01)

In [5]:
def simulate_battery_EOL(q, R0):
    batt = BatteryElectroChem(process_noise= 0.00001, measurement_noise=1)
    batt.parameters['qMobile'] = q
    batt.parameters['Ro'] = R0
    batt.parameters=update(deepcopy(batt.parameters))
    
    sim_config = {'save_freq': 2,
                  'dt': 2,
                  'threshold_keys': ['InsufficientCapacity'],
                  'print': False,
                  'progress': False}
    
    load = 1
    
    def future_loading(t, x=None):
        nonlocal load
        if x is not None:
            event_state = batt.event_state(x)
            if event_state["EOD"] > 0.95:
                load = 1
            elif event_state["EOD"] < 0.05:
                load = -1
        return batt.InputContainer({'i': load})
    
    simulated_results = batt.simulate_to_threshold(future_loading, **sim_config)
    
    return simulated_results

In [ ]:
n = 0

while n < number_of_trajectories:
    try:
        q = np.random.uniform(7500, 7600)
        R0 = np.random.uniform(0.117215-0.01, 0.117215+0.01)
        times, inputs, states, outputs, event_states = simulate_battery_EOL(q, R0)

        df = pd.DataFrame()
        df['time'] = times
        df['inputs'] = [x['i'] for x in inputs]
        df['q'] = [x['qMax'] for x in states]
        df['R0'] = [x['Ro'] for x in states]
        df['D'] = [x['D'] for x in states]
        df['temperature'] = [x['t'] for x in outputs]
        df['v'] = [x['v'] for x in outputs]
        #df.to_feather(f'{params["path"]}/{n}.feather')
        print(n)
        n += 1
        break
    except:
        print('Skipping simulation.')
        pass
    
    

Skipping simulation.


In [ ]:
import pandas as pd

data = df

data.plot(x='time', subplots=True, figsize=(15, 10), title='Battery Simulation Data')